# Violation Finder Test Notebook
This notebook verifies the `ViolationFinder` component using `desbordante` via WSL, as specified in the project requirements.

## 1. Setup & Loader Orchestration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder

def get_loader(dataset_name):
    return FileLoader(
        name=dataset_name, 
        base_path="../data",
        data_loader=DataLoader(),
        dcs_loader=DCsLoader(),
        metadata_loader=MetadataLoader(),
        data_encoder=DataEncoder(),
        dcs_encoder=DCsEncoder()
    )

## 2. Logical Tests: Datasets with NO Violations
The real-world datasets provided (e.g., `adult`, `census`, `compas`, `tax`) are expected to have NO violations.

In [ ]:
no_violation_datasets = ["adult", "census", "compas", "tax"]
for name in no_violation_datasets:
    print(f"Testing dataset: {name}")
    try:
        loader = get_loader(name)
        dataset = loader.load()
        violations = dataset.get_violations()
        print(f"  - Total violations: {len(violations)}")
        assert len(violations) == 0, f"Expected 0 violations for {name}, found {len(violations)}"
    except Exception as e:
        print(f"  - Error loading/validating {name}: {e}")

## 3. Logical Tests: Custom Dataset WITH Violations
The `custom_violations` dataset is designed to strictly fail a constraint.

In [ ]:
print("Testing dataset: custom_violations")
loader = get_loader("custom_violations")
dataset = loader.load()

print("Encoded Data Preview:")
display(dataset.data)

print("Constraints:")
for dc in dataset.dcs.constraints:
    print(dc.to_string())

violations = dataset.get_violations()
print(f"Total violations: {len(violations)}")
display(violations)

# Row 0 and 1 match A and B, but differ in C, so they violate: t1.A=t2.A & t1.B=t2.B & t1.C!=t2.C
assert len(violations) > 0, "Expected violations to be found in custom dataset!"
print("Logical tests for finding violations passed!")

## 4. Visualization & Runtime Test

In [ ]:
import time
import matplotlib.pyplot as plt
import seaborn as sns

start_time = time.time()
loader = get_loader("custom_violations")
for _ in range(5):
    v = loader.load().get_violations()
end_time = time.time()

print(f"Average runtime for custom dataset violations: {(end_time - start_time) / 5:.4f}s")

if not dataset.data.empty:
    plt.figure(figsize=(6, 3))
    sns.scatterplot(x=dataset.data.index, y=dataset.data.iloc[:, 0])
    plt.title("Custom Dataset Column 0 Distribution")
    plt.show()